# PRISM — OCR Extractor (Google Colab T4)

Extracts on-screen text from all raw videos → `ocr_text/{video_id}.txt`

**Engine:** EasyOCR (GPU-accelerated via PyTorch / CUDA on T4)  
**Speed:** ~1.5–3 s/video → 4,500 videos in **~2.5–4 hours** (fits in one 12-hour free session)

---
### Run order
1. **Cell 1** — Check GPU (must be T4)
2. **Cell 2** — Mount Google Drive
3. **Cell 3** — Install dependencies
4. **Cell 4** — Edit paths & settings
5. **Cell 5** — Text quality utilities *(run once)*
6. **Cell 6** — Frame sampling helper *(run once)*
7. **Cell 7** — OCR function *(run once)*
8. **Cell 8** — Preview 5 videos *(optional sanity check)*
9. **Cell 9** — **Full run** (resume-safe, saves after every video)
10. **Cell 10** — Summary & next steps

> ⚡ **Resume:** If the session disconnects, just re-run Cell 9 — it automatically skips videos that already have a `.txt` file.

## Cell 1 — GPU Check

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected!\n'
        'Fix: Runtime → Change runtime type → Hardware accelerator = T4 GPU'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')
print('✅ GPU ready')

GPU  : Tesla T4
VRAM : 15.6 GB
✅ GPU ready


## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## Cell 3 — Install Dependencies

In [3]:
!pip install -q easyocr opencv-python-headless tqdm
print('✅ Installs done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 32.9 MB/s eta 0:00:00
✅ Installs done


## Cell 4 — Config & Paths

> **Edit this cell** if your Google Drive folder layout is different.

In [4]:
from pathlib import Path
import torch

# ── Paths (edit if needed) ────────────────────────────────────────────────
DRIVE_BASE = Path('/content/drive/MyDrive/videostory_prism')
VIDEO_DIR  = DRIVE_BASE / 'videos' / 'raw'   # raw .mp4 files
OCR_DIR    = DRIVE_BASE / 'ocr_text'         # output .txt files (created if missing)

# ── Batch split (for multi-session runs) ─────────────────────────────────
# To process all videos:        START_IDX = 0,    END_IDX = None
# Session 1 of 3 (4500 videos): START_IDX = 0,    END_IDX = 1500
# Session 2 of 3:               START_IDX = 1500, END_IDX = 3000
# Session 3 of 3:               START_IDX = 3000, END_IDX = 4500
START_IDX = 3000
END_IDX   = None   # None = process to the end

# ── OCR settings ──────────────────────────────────────────────────────────
# Frames per second to sample from each video.
# 0.5 = 1 frame every 2 sec  → good for most content, fast.
# 1.0 = 1 frame per sec      → better coverage for news / fast-cut videos.
OCR_FPS       = 0.5

TOP_K_LINES   = 5     # best text lines to keep per video
MAX_CHARS     = 1200  # max characters stored per .txt file
MIN_LINE_LEN  = 6     # lines shorter than this are discarded
LANGUAGES     = ['en'] # EasyOCR languages (add e.g. 'fr' for French)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Create output dir ────────────────────────────────────────────────────
OCR_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video dir : {VIDEO_DIR}')
print(f'OCR dir   : {OCR_DIR}')
print(f'Device    : {DEVICE}')
print(f'FPS       : {OCR_FPS}')
print(f'Slice     : [{START_IDX} : {END_IDX}]')

Video dir : /content/drive/MyDrive/videostory_prism/videos/raw
OCR dir   : /content/drive/MyDrive/videostory_prism/ocr_text
Device    : cuda
FPS       : 0.5
Slice     : [3000 : None]


## Cell 5 — Text Quality Utilities

Scores each OCR line for usefulness — filters garbled text, URLs, timestamps, watermarks.  
*(Ported from `PRISM_BERT_Extract.ipynb`)*

In [5]:
import re, math

def score_line(line: str) -> float:
    """
    Score a single OCR text line. Returns 0.0 for garbage.
    Score = alpha_word_ratio * log(length + 1).
    """
    line = line.strip()
    if len(line) < MIN_LINE_LEN:
        return 0.0
    words   = line.split()
    alpha_w = [w for w in words if re.match(r'^[A-Za-z]{2,}$', w)]
    alpha_r = len(alpha_w) / max(len(words), 1)
    if re.search(r'(https?://|www\.|\.(com|net|org|co\.))', line.lower()):
        return 0.0   # URLs / domain names
    if alpha_r < 0.50:
        return 0.0   # mostly numbers / symbols
    if re.match(r'^[\d\s\W]{5,}$', line):
        return 0.0   # pure numeric/symbol strings
    return alpha_r * math.log(len(line) + 1)


def select_best_lines(frame_texts: list, top_k: int = TOP_K_LINES,
                      max_chars: int = MAX_CHARS) -> str:
    """
    Flatten OCR results from multiple frames, deduplicate, score,
    and return the top-k most informative lines as one clean string.
    """
    all_lines = []
    for block in frame_texts:
        all_lines.extend(ln.strip() for ln in re.split(r'[\n|]+', block) if ln.strip())

    # Case-insensitive deduplication
    seen, unique = set(), []
    for ln in all_lines:
        key = ln.lower()
        if key not in seen:
            seen.add(key)
            unique.append(ln)

    scored = [(score_line(ln), ln) for ln in unique]
    picked = [ln for s, ln in sorted(scored, reverse=True) if s > 0][:top_k]
    return ' '.join(picked)[:max_chars].strip()


# Quick test
test_lines = [
    'CNN Breaking News — President signs bill',
    'DNOOUVB DNDBUYB N08UV',
    '25359 82:253-59 ND VEF',
    'How to remove rust off a car using basic tools.',
    'https://cnn.com/live'
]
print('Line scoring test:')
for l in test_lines:
    print(f'  {score_line(l):.2f}  |  {l[:60]}')

Line scoring test:
  3.18  |  CNN Breaking News — President signs bill
  2.06  |  DNOOUVB DNDBUYB N08UV
  1.57  |  25359 82:253-59 ND VEF
  3.10  |  How to remove rust off a car using basic tools.
  0.00  |  https://cnn.com/live


## Cell 6 — Frame Sampling Helper

In [6]:
import cv2, numpy as np

def sample_frames(video_path, target_fps: float) -> list:
    """
    Extract frames at ~target_fps uniformly from video.
    Returns list of RGB numpy arrays. Returns [] on corrupt video.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    native_fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    duration_s   = total_frames / native_fps

    n         = max(1, min(int(duration_s * target_fps), total_frames))
    positions = [int(total_frames * (i + 0.5) / n) for i in range(n)]

    frames = []
    for pos in positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, min(pos, total_frames - 1))
        ret, frame = cap.read()
        if ret and frame is not None:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    cap.release()
    return frames


print('✅ Frame sampler ready')

✅ Frame sampler ready


## Cell 7 — Load EasyOCR & Define Extraction Function

> First run downloads ~100 MB of model weights (cached after that).

In [7]:
import easyocr

print(f'Loading EasyOCR (gpu={DEVICE=="cuda"}) ...')
ocr_reader = easyocr.Reader(LANGUAGES, gpu=(DEVICE == 'cuda'), verbose=False)
print('✅ EasyOCR ready')


def extract_ocr_for_video(video_path, target_fps: float = OCR_FPS) -> str:
    """
    Sample frames → EasyOCR each frame → deduplicate + clean → return text.
    Returns '' if no useful text found (video-only content).
    """
    frames = sample_frames(video_path, target_fps)
    if not frames:
        return ''

    raw_texts = []
    for frame in frames:
        try:
            # detail=0 = plain strings; paragraph=True merges nearby lines
            results = ocr_reader.readtext(frame, detail=0, paragraph=True)
            if results:
                raw_texts.append('\n'.join(str(r) for r in results))
        except Exception:
            pass

    return select_best_lines(raw_texts)


print('✅ extract_ocr_for_video() ready')

Loading EasyOCR (gpu=True) ...


✅ EasyOCR ready
✅ extract_ocr_for_video() ready


## Cell 8 — Preview (Optional)

Run on a few videos to verify OCR output before committing to the full run.

In [8]:
PREVIEW_N = 5   # how many videos to preview

video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
all_videos = sorted(
    [f for f in VIDEO_DIR.iterdir() if f.suffix.lower() in video_exts],
    key=lambda f: (0, int(f.stem.split('_')[-1]), f.stem)
    if f.stem.split('_')[-1].isdigit() else (1, 0, f.stem)
)
print(f'Total videos found: {len(all_videos)}')
print(f'\n🔍 Previewing first {PREVIEW_N} videos:\n')

for vf in all_videos[:PREVIEW_N]:
    text = extract_ocr_for_video(vf)
    preview = text[:120].replace('\n', ' ') if text else '(no text found)'
    print(f'  {vf.stem:<20}  →  {preview}')

Total videos found: 4500

🔍 Previewing first 5 videos:

  vs_3                  →  Although you are so comfortably. are You looking at the scene in front of the eyes? or d0 you see it? was walking in the
  vs_4                  →  Annde tlnndl 09/01/2013 11:56:36 AM PLAYE 09/01/2013 11:56:37 AM PLAY 09/01/2013 11:56:33 AM PLAY 09/01/2013 11:56:31 AM
  vs_5                  →  TheHammerPhone com For more, go to:
  vs_6                  →  Only Uhich? uses potatoes and nails to test 115 microwaves for fire safety Only Uhich? uses baked on egg and spinach to 
  vs_7                  →  (no text found)


## Cell 9 — Full Run

- **Resume-safe**: each video is saved immediately as its own `.txt` file.  
  If the session disconnects, just re-run this cell — already-done videos are skipped.
- **Slice**: controlled by `START_IDX` / `END_IDX` in Cell 4.

In [9]:
import time
from tqdm.auto import tqdm

# ── discover videos ───────────────────────────────────────────────────────
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

def _sort_key(f):
    parts = f.stem.split('_')
    return (0, int(parts[-1]), f.stem) if parts[-1].isdigit() else (1, 0, f.stem)

all_videos = sorted(
    [f for f in VIDEO_DIR.iterdir() if f.suffix.lower() in video_exts],
    key=_sort_key
)

batch = all_videos[START_IDX:END_IDX]
print(f'Total videos : {len(all_videos)}')
print(f'Batch slice  : [{START_IDX}:{END_IDX}]  → {len(batch)} videos')

# ── skip already done ─────────────────────────────────────────────────────
done_ids  = {f.stem for f in OCR_DIR.glob('*.txt')}
remaining = [v for v in batch if v.stem not in done_ids]
skipped   = len(batch) - len(remaining)
print(f'Already done : {skipped}  (skipping)')
print(f'To process   : {len(remaining)}')

if not remaining:
    print('\n✅ All videos in this slice already processed!')
else:
    est_s = len(remaining) * 2.0   # ~2 sec/video on T4 GPU at 0.5 FPS
    print(f'ETA          : ~{est_s/3600:.1f}h  (GPU estimate at {OCR_FPS} FPS)\n')

    # ── main loop ──────────────────────────────────────────────────────────
    t0           = time.time()
    done_count   = 0
    empty_count  = 0
    failed_count = 0

    for i, vid_path in enumerate(tqdm(remaining, desc='OCR')):
        vid      = vid_path.stem
        out_file = OCR_DIR / f'{vid}.txt'

        try:
            text = extract_ocr_for_video(vid_path)
            out_file.write_text(text, encoding='utf-8')
            done_count += 1
            if not text:
                empty_count += 1

        except Exception as e:
            tqdm.write(f'  ⚠️  Error on {vid}: {e}')
            out_file.write_text('', encoding='utf-8')  # mark as processed
            failed_count += 1

        # Progress every 10 videos
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            rate    = (i + 1) / elapsed
            eta_s   = (len(remaining) - i - 1) / rate if rate > 0 else 0
            eta_str = f'{eta_s/3600:.1f}h' if eta_s > 3600 else f'{eta_s/60:.0f}m'
            tqdm.write(f'  [{vid}]  rate={rate:.1f} vid/s  ETA: {eta_str}')

    elapsed = time.time() - t0
    print(f'\nDone in {elapsed/3600:.2f}h')
    print(f'Processed : {done_count}  |  With text : {done_count - empty_count}  |  '
          f'Empty : {empty_count}  |  Errors : {failed_count}')

Total videos : 4500
Batch slice  : [3000:None]  → 1500 videos
Already done : 125  (skipping)
To process   : 1375
ETA          : ~0.8h  (GPU estimate at 0.5 FPS)



OCR:   0%|          | 0/1375 [00:00<?, ?it/s]

  [vs_12853]  rate=0.1 vid/s  ETA: 2.6h
  [vs_12867]  rate=0.2 vid/s  ETA: 2.2h
  [vs_12883]  rate=0.2 vid/s  ETA: 2.0h
  [vs_12900]  rate=0.2 vid/s  ETA: 1.9h
  [vs_12925]  rate=0.2 vid/s  ETA: 1.9h
  [vs_12941]  rate=0.2 vid/s  ETA: 1.9h
  [vs_12961]  rate=0.2 vid/s  ETA: 1.8h
  [vs_12985]  rate=0.2 vid/s  ETA: 1.8h
  [vs_13009]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13030]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13043]  rate=0.2 vid/s  ETA: 1.8h
  [vs_13072]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13095]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13114]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13134]  rate=0.2 vid/s  ETA: 1.7h
  [vs_13151]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13167]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13187]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13204]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13221]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13235]  rate=0.2 vid/s  ETA: 1.6h
  [vs_13261]  rate=0.2 vid/s  ETA: 1.5h
  [vs_13282]  rate=0.2 vid/s  ETA: 1.5h
  [vs_13303]  rate=0.2 vid/s  ETA: 1.5h
  [vs_13335]  rate=0.2 vid/s  ETA: 1.5h


## Cell 10 — Summary & Quality Distribution

In [10]:
# Count all .txt files produced so far
all_txt = list(OCR_DIR.glob('*.txt'))
with_text  = [f for f in all_txt if f.read_text(encoding='utf-8').strip()]
empty_txt  = [f for f in all_txt if not f.read_text(encoding='utf-8').strip()]

print('=' * 50)
print('OCR Extraction Summary')
print('=' * 50)
print(f'Total .txt files  : {len(all_txt)}')
print(f'With text         : {len(with_text)}  ({100*len(with_text)/max(len(all_txt),1):.1f}%)')
print(f'Empty (no text)   : {len(empty_txt)}  ({100*len(empty_txt)/max(len(all_txt),1):.1f}%)')
print(f'Output dir        : {OCR_DIR}')
print('=' * 50)

if empty_txt:
    print(f'\n💡 {len(empty_txt)} videos have no on-screen text (video-only).')
    print('   Gemini 2.5 Flash will still generate titles using visual content.')
    print('   Example empty videos:', [f.stem for f in empty_txt[:5]])

print('\n✅ Next step: run run_gt_gemini.py to generate ground truth titles.')

OCR Extraction Summary
Total .txt files  : 4152
With text         : 2337  (56.3%)
Empty (no text)   : 1815  (43.7%)
Output dir        : /content/drive/MyDrive/videostory_prism/ocr_text

💡 1815 videos have no on-screen text (video-only).
   Gemini 2.5 Flash will still generate titles using visual content.
   Example empty videos: ['vs_2409', 'vs_2410', 'vs_2415', 'vs_2421', 'vs_2425']

✅ Next step: run run_gt_gemini.py to generate ground truth titles.
